### Connect postgresql database

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

# 数据库配置
username = "XXXXXX"
password = "YYYYYY"
host = "localhost"
port = 5432
database = "eyewear-data"

# 创建连接
engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)

# 查询数据
sql = """
SELECT
    o.order_id,
    o.customer_id,
    o.order_date,
    o.order_status,
    o.total_price_before_tax,
    oi.product_id,
    oi.quantity,
    oi.product_name,
    oi.unit_price,
    oi.line_price_before_tax,
    oi.is_free_gift,
    pa.campaign_id,
    p.product_name AS productinfo_product_name,
    p.product_m3_code,
    p.cost_price,
    s.store_id,
    s.store_region,
    s.store_name
FROM "Order" o
JOIN "OrderItem" oi
    ON o.order_id = oi.order_id
JOIN "PromotionActivity" pa
    ON o.campaign_id = pa.campaign_id
JOIN "ProductInfo" p
    ON oi.product_id = p.product_id
JOIN "StoreInfo" s
    ON o.store_id = s.store_id
WHERE o.order_status IN ('Completed', 'Shipped');
"""

df_order_completed_shipped = pd.read_sql(sql, engine)

# 查看数据
df_order_completed_shipped

### Calculate cost per year 

In [ ]:
# 确保 order_date 是 datetime
df_order_completed_shipped["order_date"] = pd.to_datetime(df_order_completed_shipped["order_date"])

# 增加年份列
df_order_completed_shipped["order_year"] = df_order_completed_shipped["order_date"].dt.year

# 计算每行订单项的成本
df_order_completed_shipped["order_cost"] = (
    df_order_completed_shipped["cost_price"] * df_order_completed_shipped["quantity"]
)

# 按 order_id + order_year 汇总每张订单的总成本
df_cost_per_product = (
    df_order_completed_shipped
    .groupby(["order_year", "product_id", "product_m3_code"], as_index=False)["order_cost"]
    .sum()
)

df_cost_per_product

### Calculate `total_price_before_tax` per order 

In [ ]:
# Order表事先已计算好了`total_price_before_tax`了

### Calculate `gross profit` and `gross margin` per order 

In [ ]:
# 先取每张订单的收入（去重订单行）
df_revenue_per_product = (
    df_order_completed_shipped
    .groupby(["order_year", "product_id", "product_m3_code"], as_index=False)
    .agg(total_revenue=("line_price_before_tax", "sum"))
)

df_cost_per_product = (
    df_order_completed_shipped
    .groupby(["order_year", "product_id", "product_m3_code"], as_index=False)
    .agg(order_cost=("order_cost", "sum"))
)

# 合并成本和收入
df_gross_per_product = df_cost_per_product.merge(
    df_revenue_per_product,
    on=["order_year", "product_id", "product_m3_code"],
    how="left"
)

# 计算毛利和毛利率
df_gross_per_product["gross_profit"] = (
    df_gross_per_product["total_revenue"] - df_gross_per_product["order_cost"]
)
df_gross_per_product["gross_margin"] = (
    df_gross_per_product["gross_profit"] / df_gross_per_product["total_revenue"].replace(0, pd.NA)
)

# 按年份汇总成本、收入、毛利
df_product = (
    df_gross_per_product
    .groupby(["product_id", "product_m3_code"], as_index=False)
    .agg(
        order_cost=("order_cost", "sum"),
        total_revenue=("total_revenue", "sum"),
        gross_profit=("gross_profit", "sum")
    )
)

df_product["gross_margin"] = df_product["gross_profit"] / df_product["total_revenue"].replace(0, pd.NA)

df_product

In [ ]:
# 关闭数据库连接
engine.dispose()